# Pilot: 2019-20 Injury Data Validation
- Data from Kaggle (https://www.kaggle.com/datasets/ghopkins/nba-injuries-2010-2018)
- Goal: 
1) Collect and evaluate 2019-2020 season data. Identifying duplicate episodes. 
2) Compare against Official NBA data using nba_api.
3) Conclude whether this data source will be adequate for our main injury data source. 

## 1. Load the injury dataset

In [8]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../Data/Raw/injuries_2010-2020.csv")
df["Date"] = pd.to_datetime(df["Date"])

rows, cols = df.shape
print(f"Dataset Loaded: Rows: {rows}, Columns: {cols}")
print("Full dataset ends:", df["Date"].max())
display(df.head())

pilot_df = df.loc[df["Date"].between("2019-10-22", "2020-08-14")].copy()


Dataset Loaded: Rows: 27105, Columns: 5
Full dataset ends: 2020-10-06 00:00:00


,Date,Team,Acquired,Relinquished,Notes
0,2010-10-03,Bulls,NaN,Carlos Boozer,fractured bone in right pinky finger (out inde...
1,2010-10-06,Pistons,NaN,Jonas Jerebko,torn right Achilles tendon (out indefinitely)
2,2010-10-06,Pistons,NaN,Terrico White,broken fifth metatarsal in right foot (out ind...
3,2010-10-08,Blazers,NaN,Jeff Ayres,torn ACL in right knee (out indefinitely)
4,2010-10-08,Nets,NaN,Troy Murphy,strained lower back (out indefinitely)


## 2. Select the 2019-20 regular season

In [9]:
pilot_df.head()
rows, cols = pilot_df.shape
print(f"Dataset Loaded: Rows: {rows}, Columns: {cols}")
print("Season records:", len(pilot_df))
print("\n == NA VALUES ==")
display(pilot_df.isna().sum())
pilot_df.head()

Dataset Loaded: Rows: 2210, Columns: 5
Season records: 2210

 == NA VALUES ==


Date               0
Team               0
Acquired        1307
Relinquished     905
Notes              0
dtype: int64

,Date,Team,Acquired,Relinquished,Notes
24807,2019-10-22,Blazers,NaN,Pau Gasol,left foot injury (DTD)
24808,2019-10-22,Bulls,NaN,Chandler Hutchison,strained left hamstring (DTD)
24809,2019-10-22,Celtics,NaN,Romeo Langford,sore right knee (DTD)
24810,2019-10-22,Celtics,NaN,Tacko Fall,concussion (DTD)
24811,2019-10-22,Clippers,NaN,Paul George,placed on IL with right shoulder injury


## 3. Identify ankle-sprain reports

In [10]:

# Restore the original season records before looking up returns.
pilot_df = df.loc[
    df["Date"].between("2019-10-22", "2020-08-14")
].copy()

ankle_sprains = pilot_df[
pilot_df["Notes"].str.contains("ankle", case=False, na=False)
    & pilot_df["Notes"].str.contains("sprain", case=False, na=False)
].copy()

print("Candidate records:", len(ankle_sprains))
display(ankle_sprains)

Candidate records: 131


,Date,Team,Acquired,Relinquished,Notes
24812,2019-10-22,Clippers,NaN,Rodney McGruder,placed on IL with sprained right ankle
24825,2019-10-22,Suns,NaN,Ty Jerome,sprained right ankle (out indefinitely)
24838,2019-10-23,Cavaliers,NaN,John Henson,placed on IL with sprained right ankle
24849,2019-10-23,Knicks,NaN,Mitchell Robinson,placed on IL with sprained right ankle
24852,2019-10-23,Mavericks,NaN,Ryan Broekhoff,placed on IL with sprained right ankle
...,...,...,...,...,...
26869,2020-08-06,Heat,NaN,Goran Dragic,placed on IL with sprained left ankle
26917,2020-08-09,Wizards,NaN,Shabazz Napier,placed on IL with sprained right ankle (out fo...
26927,2020-08-10,Nets,NaN,Jarrett Allen,sprained left ankle (DTD)
26968,2020-08-12,Celtics,NaN,Jayson Tatum,placed on IL with sprained left ankle


## 4. Combine repeated reports into injury episodes

In [11]:
ankle_sprains["Player"] = ankle_sprains["Relinquished"].str.strip()

player_counts = ankle_sprains["Player"].value_counts()
display(player_counts[player_counts > 1])

Player
Deandre Ayton         4
Daniel Theis          3
Jahlil Okafor         3
Moe Wagner            3
Luka Doncic           3
Ty Jerome             2
John Henson           2
Trae Young            2
Myles Turner          2
Serge Ibaka           2
De'Aaron Fox          2
Landry Shamet         2
Rudy Gobert           2
Romeo Langford        2
Naz Mitrou-Long       2
Dewan Hernandez       2
Glenn Robinson III    2
Nerlens Noel          2
Draymond Green        2
Wendell Carter Jr.    2
Abdel Nader           2
R.J. Barrett          2
Jaylen Brown          2
De'Andre Hunter       2
P.J. Washington       2
Dario Saric           2
Meyers Leonard        2
Mario Hezonja         2
Luke Kornet           2
Ky Bowman             2
Derrick Rose          2
Seth Curry            2
Eric Gordon           2
Name: count, dtype: int64

In [12]:
# Set up columns for each sprain report.
ankle_sprains["Out Date"] = ankle_sprains["Date"]
ankle_sprains["Cleared Date"] = pd.NaT
ankle_sprains["Injury"] = "ankle sprain"
ankle_sprains["Reports"] = 1
ankle_sprains["episode_id"] = ankle_sprains.index # Start with each report as its own episode

for index, row in ankle_sprains.iterrows():
    # Get this player's return dates.
    return_dates = pilot_df.loc[
        (pilot_df["Acquired"].str.strip() == row["Player"])
        & pilot_df["Notes"].str.contains(
            "activated from IL|returned to lineup",
            case=False,
            na=False
        ),
        "Date"
    ]

    injury_date = row["Date"]

    # Skip reports where the player returned the same day
    same_day_return = (return_dates == injury_date).any()
    if not same_day_return:
        # Use the first return after the injury as the cleared date.
        later_returns = return_dates[return_dates > injury_date]
        cleared_date = later_returns.min()
        ankle_sprains.loc[index, "Cleared Date"] = cleared_date


# Reports for the same player with the same return date share an ID.
# Reports without a return date stay separate.
for group, reports in ankle_sprains.groupby(["Player", "Cleared Date"]):
    ankle_sprains.loc[reports.index, "episode_id"] = reports.index[0]

# Combine reports into one row per estimated episode.
pilot_df = ankle_sprains.groupby("episode_id", as_index=False).agg({
    "Team": "first",
    "Player": "first",
    "Out Date": "min", # Earliest report date
    "Cleared Date": "first", # Same for every report in the episode
    "Injury": "first", 
    "Reports": "sum" # Number of reports combined into this episode
})

# Show all episodes.
display(pilot_df)
print("Episodes with multiple reports:")
display(pilot_df[pilot_df["Reports"] > 1])

,episode_id,Team,Player,Out Date,Cleared Date,Injury,Reports
0,24812,Clippers,Rodney McGruder,2019-10-22,2019-10-30,ankle sprain,1
1,24825,Suns,Ty Jerome,2019-10-22,2019-12-02,ankle sprain,2
2,24838,Cavaliers,John Henson,2019-10-23,2019-10-26,ankle sprain,1
3,24849,Knicks,Mitchell Robinson,2019-10-23,2019-10-25,ankle sprain,1
4,24852,Mavericks,Ryan Broekhoff,2019-10-23,2019-10-25,ankle sprain,1
...,...,...,...,...,...,...,...
101,26869,Heat,Goran Dragic,2020-08-06,2020-08-10,ankle sprain,1
102,26917,Wizards,Shabazz Napier,2020-08-09,NaT,ankle sprain,1
103,26927,Nets,Jarrett Allen,2020-08-10,2020-08-12,ankle sprain,1
104,26968,Celtics,Jayson Tatum,2020-08-12,NaT,ankle sprain,1


Episodes with multiple reports:


,episode_id,Team,Player,Out Date,Cleared Date,Injury,Reports
1,24825,Suns,Ty Jerome,2019-10-22,2019-12-02,ankle sprain,2
12,24945,Pacers,Myles Turner,2019-10-31,2019-11-16,ankle sprain,2
16,25044,Raptors,Serge Ibaka,2019-11-09,2019-12-01,ankle sprain,2
17,25061,Kings,De'Aaron Fox,2019-11-11,2019-12-17,ankle sprain,2
19,25078,Clippers,Landry Shamet,2019-11-12,2019-12-14,ankle sprain,2
22,25117,Pelicans,Jahlil Okafor,2019-11-14,2019-11-17,ankle sprain,2
28,25226,Jazz,Rudy Gobert,2019-11-23,2019-11-27,ankle sprain,2
29,25259,Celtics,Romeo Langford,2019-11-25,2019-12-11,ankle sprain,2
36,25528,Wizards,Moe Wagner,2019-12-13,2020-02-02,ankle sprain,2
37,25549,Mavericks,Luka Doncic,2019-12-15,2019-12-26,ankle sprain,2


In [13]:
pilot_df.to_csv(
    "../Data/Processed/injury_episodes_2019_20.csv",
    index=False
)

## 5. Validate injury dates with NBA game logs
- *AI Generated* 
- Model: OpenAI Astra Medium 
- Conversation Date: 11/16/2026 

In [20]:
from nba_api.stats.endpoints import playergamelogs, leaguegamelog


# ============================================================
# STEP 1: Copy your existing injury episodes
# ============================================================

injury_date_validation_df = pilot_df.copy()

required_columns = ["Player", "Team", "Out Date"]

for column in required_columns:
    if column not in injury_date_validation_df.columns:
        raise ValueError(
            f"Your pilot_df is missing the '{column}' column."
        )

# Rename Out Date to Injury Date.
injury_date_validation_df = injury_date_validation_df.rename(
    columns={"Out Date": "Injury Date"}
)

# Make sure Injury Date is stored as a date.
injury_date_validation_df["Injury Date"] = pd.to_datetime(
    injury_date_validation_df["Injury Date"]
)


# ============================================================
# STEP 2: Download the 2019-20 NBA game logs
# ============================================================

print("Downloading NBA player game logs...")

nba_player_games = playergamelogs.PlayerGameLogs(
    season_nullable="2019-20",
    season_type_nullable="Regular Season",
    timeout=60
).get_data_frames()[0]

print("Downloading NBA team game logs...")

nba_team_games = leaguegamelog.LeagueGameLog(
    season="2019-20",
    season_type_all_star="Regular Season",
    player_or_team_abbreviation="T",
    sorter="DATE",
    direction="ASC",
    timeout=60
).get_data_frames()[0]

print("NBA data downloaded.")


# ============================================================
# STEP 3: Prepare the NBA data
# ============================================================

# Convert the NBA game dates to datetime values.
nba_player_games["GAME_DATE"] = pd.to_datetime(
    nba_player_games["GAME_DATE"]
)

nba_team_games["GAME_DATE"] = pd.to_datetime(
    nba_team_games["GAME_DATE"]
)

# Convert GAME_ID to text so both dataframes use the same type.
nba_player_games["GAME_ID"] = (
    nba_player_games["GAME_ID"].astype(str)
)

nba_team_games["GAME_ID"] = (
    nba_team_games["GAME_ID"].astype(str)
)


# ============================================================
# STEP 4: Convert team names to NBA abbreviations
#
# Example:
# Bulls becomes CHI
# Lakers becomes LAL
# ============================================================

team_abbreviations = {
    "76ers": "PHI",
    "Bucks": "MIL",
    "Bulls": "CHI",
    "Cavaliers": "CLE",
    "Celtics": "BOS",
    "Clippers": "LAC",
    "Grizzlies": "MEM",
    "Hawks": "ATL",
    "Heat": "MIA",
    "Hornets": "CHA",
    "Jazz": "UTA",
    "Kings": "SAC",
    "Knicks": "NYK",
    "Lakers": "LAL",
    "Magic": "ORL",
    "Mavericks": "DAL",
    "Nets": "BKN",
    "Nuggets": "DEN",
    "Pacers": "IND",
    "Pelicans": "NOP",
    "Pistons": "DET",
    "Raptors": "TOR",
    "Rockets": "HOU",
    "Spurs": "SAS",
    "Suns": "PHX",
    "Thunder": "OKC",
    "Timberwolves": "MIN",
    "Blazers": "POR",
    "Trail Blazers": "POR",
    "Warriors": "GSW",
    "Wizards": "WAS"
}

injury_date_validation_df["Team Abbreviation"] = (
    injury_date_validation_df["Team"].map(team_abbreviations)
)

# Display any team names that were not matched.
unmatched_teams = injury_date_validation_df.loc[
    injury_date_validation_df["Team Abbreviation"].isna(),
    "Team"
].drop_duplicates()

if len(unmatched_teams) > 0:
    print("\nThese team names were not matched:")
    print(unmatched_teams.tolist())


# ============================================================
# STEP 5: Clean player names for matching
#
# For example:
# P.J. Washington becomes pj washington
# ============================================================

def clean_player_name(name):
    name = str(name).lower()
    name = name.replace(".", "")
    name = name.replace("'", "")
    name = name.replace("-", " ")
    name = " ".join(name.split())
    return name


injury_date_validation_df["Name Key"] = (
    injury_date_validation_df["Player"].apply(
        clean_player_name
    )
)

nba_player_games["Name Key"] = (
    nba_player_games["PLAYER_NAME"].apply(
        clean_player_name
    )
)


# ============================================================
# STEP 6: Validate each reported Injury Date
# ============================================================

last_appearances = []
next_team_games = []
days_until_next_game = []
played_next_game_results = []
validation_results = []

for index, injury in injury_date_validation_df.iterrows():

    player_name = injury["Name Key"]
    team = injury["Team Abbreviation"]
    injury_date = injury["Injury Date"]

    # Find every NBA game played by this player.
    player_games = nba_player_games[
        nba_player_games["Name Key"] == player_name
    ].sort_values("GAME_DATE")

    # --------------------------------------------------------
    # Check whether the player name matched the NBA data.
    # --------------------------------------------------------

    if player_games.empty:
        last_appearances.append(pd.NaT)
        next_team_games.append(pd.NaT)
        days_until_next_game.append(pd.NA)
        played_next_game_results.append(pd.NA)
        validation_results.append(
            "Player name not matched"
        )
        continue

    # --------------------------------------------------------
    # Check whether the team name matched.
    # --------------------------------------------------------

    if pd.isna(team):
        last_appearances.append(pd.NaT)
        next_team_games.append(pd.NaT)
        days_until_next_game.append(pd.NA)
        played_next_game_results.append(pd.NA)
        validation_results.append(
            "Team name not matched"
        )
        continue

    # --------------------------------------------------------
    # Find the player's last NBA appearance on or before the
    # reported Injury Date.
    # --------------------------------------------------------

    games_before_injury = player_games[
        player_games["GAME_DATE"] <= injury_date
    ]

    if games_before_injury.empty:
        last_appearance = pd.NaT
    else:
        last_appearance = (
            games_before_injury.iloc[-1]["GAME_DATE"]
        )

    last_appearances.append(last_appearance)

    # --------------------------------------------------------
    # Find the team's first scheduled game after the reported
    # Injury Date.
    #
    # We use > instead of >= because the injury data does not
    # provide a time of day. A player may have been injured
    # during a game played on the Injury Date.
    # --------------------------------------------------------

    later_team_games = nba_team_games[
        (nba_team_games["TEAM_ABBREVIATION"] == team)
        & (nba_team_games["GAME_DATE"] > injury_date)
    ].sort_values("GAME_DATE")

    if later_team_games.empty:
        next_team_games.append(pd.NaT)
        days_until_next_game.append(pd.NA)
        played_next_game_results.append(pd.NA)
        validation_results.append(
            "No later regular-season team game"
        )
        continue

    next_game = later_team_games.iloc[0]
    next_game_date = next_game["GAME_DATE"]
    next_game_id = next_game["GAME_ID"]

    next_team_games.append(next_game_date)

    days_until_next_game.append(
        (next_game_date - injury_date).days
    )

    # --------------------------------------------------------
    # Check whether the player appeared in the team's next
    # scheduled game.
    # --------------------------------------------------------

    player_appeared = (
        player_games["GAME_ID"] == next_game_id
    ).any()

    played_next_game_results.append(player_appeared)

    if player_appeared:
        validation_results.append(
            "Played in next game"
        )
    else:
        validation_results.append(
            "Supports Injury Date: missed next game"
        )


# ============================================================
# STEP 7: Add the results to the dataframe
# ============================================================

injury_date_validation_df["Last NBA Appearance"] = (
    last_appearances
)

injury_date_validation_df["Next Team Game"] = (
    next_team_games
)

injury_date_validation_df["Days Until Next Game"] = (
    pd.array(
        days_until_next_game,
        dtype="Int64"
    )
)

injury_date_validation_df["Played in Next Game"] = (
    played_next_game_results
)

injury_date_validation_df["Injury Date Validation"] = (
    validation_results
)


# ============================================================
# STEP 8: Display the results
# ============================================================

result_columns = [
    "Player",
    "Team",
    "Injury Date",
    "Last NBA Appearance",
    "Next Team Game",
    "Days Until Next Game",
    "Played in Next Game",
    "Injury Date Validation"
]

display(
    injury_date_validation_df[result_columns]
)

print("\nInjury Date validation summary:")

print(
    injury_date_validation_df[
        "Injury Date Validation"
    ].value_counts()
)

print("\nOriginal injury episodes:", len(pilot_df))

print(
    "Validated injury episodes:",
    len(injury_date_validation_df)
)


# ============================================================
# STEP 9: Save the results
# ============================================================

injury_date_validation_df[result_columns].to_csv(
    "../Data/Processed/injury_date_validation_2019_20.csv",
    index=False
)

# Save the NBA data so you have a local copy.
nba_player_games.to_csv(
    "../Data/Raw/nba_player_games_2019_20.csv",
    index=False
)

nba_team_games.to_csv(
    "../Data/Raw/nba_team_games_2019_20.csv",
    index=False
)

print("\nInjury Date validation files saved successfully.")

NBA data downloaded.


,Player,Team,Injury Date,Last NBA Appearance,Next Team Game,Days Until Next Game,Played in Next Game,Injury Date Validation
0,Rodney McGruder,Clippers,2019-10-22,NaT,2019-10-24,2,False,Supports Injury Date: missed next game
1,Ty Jerome,Suns,2019-10-22,NaT,2019-10-23,1,False,Supports Injury Date: missed next game
2,John Henson,Cavaliers,2019-10-23,NaT,2019-10-26,3,True,Played in next game
3,Mitchell Robinson,Knicks,2019-10-23,NaT,2019-10-25,2,True,Played in next game
4,Ryan Broekhoff,Mavericks,2019-10-23,NaT,2019-10-25,2,False,Supports Injury Date: missed next game
...,...,...,...,...,...,...,...,...
101,Goran Dragic,Heat,2020-08-06,2020-08-04,2020-08-08,2,False,Supports Injury Date: missed next game
102,Shabazz Napier,Wizards,2020-08-09,2020-08-07,2020-08-11,2,False,Supports Injury Date: missed next game
103,Jarrett Allen,Nets,2020-08-10,2020-08-09,2020-08-11,1,False,Supports Injury Date: missed next game
104,Jayson Tatum,Celtics,2020-08-12,2020-08-11,2020-08-13,1,False,Supports Injury Date: missed next game



Injury Date validation summary:
Injury Date Validation
Supports Injury Date: missed next game    71
Played in next game                       24
Player name not matched                    8
No later regular-season team game          3
Name: count, dtype: int64

Original injury episodes: 106
Validated injury episodes: 106

Injury Date validation files saved successfully.
